# Block 2 - Assignment 2
Bogdan Filipchuk, Yunhee Choi, Gregorio A. Rolli, Artem Hrynov

## 1. Types of calculations AiiDAlab provides


The adsorption workflow provides three types of calculations:

- **Pore analysis:** computes geometric properties (density, accessible surface area, accessible
  probe-occupiable volume and porosity) for a specified probe radius.

- **Test-particle insertion:** samples the insertion of a molecule
  into the framework to calculate the Henry coefficient and adsorption
  energetics at infinite dilution.

- **Grand Canonical Monte Carlo (GCMC):** samples adsorption at fixed
  temperature, volume and chemical potential, allowing the number of
  adsorbed molecules to fluctuate. Calculations at different pressures
  provide the adsorption isotherm.

## 2. Properties of the material

- Density: $0.576983\ \mathrm{g\,cm^{-3}}$
- Accessible surface area: $3964.65\ \mathrm{\AA^2}$
- Accessible probe-occupiable volume: $13737.4\ \mathrm{\AA^3}$
- Unit-cell volume: $17726.2\ \mathrm{\AA^3}$

The accessible porosity is calculated as

$$
\varepsilon
= \frac{V_{\mathrm{POAV}}}{V_{\mathrm{cell}}}
= \frac{13737.4}{17726.2}
= 0.77498.
$$

So, the accessible porosity is approximately $77.50\%$.

## 3. Henry coefficients

The computed Henry coefficients for $CO_2$ and $CH_4$ are $4.95566\cdot 10^{-6}$ $\frac{mol}{kg\cdot Pa}$ and $1.12067\cdot 10^{-6}$ $\frac{mol}{kg\cdot Pa}$, respectively.

Here, the loading is proportional to the gas partial
pressure: $q_i = K_{H,i}p_i$. The limiting adsorption selectivity is

$$
S_{\mathrm{CO_2/CH_4}}^{0}
= \frac{K_{H,\mathrm{CO_2}}}{K_{H,\mathrm{CH_4}}}
= 4.42
$$

IRMOF-1 thus preferentially adsorbs CO₂ at low pressure. 

## 4. Isotherms calculations


In [ ]:
import ast
import csv
import matplotlib.pyplot as plt

plt.rcdefaults()

FILES = [
    ("ch4_results.csv", r"$\mathrm{CH}_4$ isotherm"),
    ("co2_results.csv", r"$\mathrm{CO}_2$ isotherm"),
]

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
})

colors = [plt.colormaps["magma"](0.25), plt.colormaps["magma"](0.75)]
markers = ["o", "s"]

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

for (filename, label), color, marker in zip(FILES, colors, markers):
    with open(filename, newline="") as file:
        data = {
            row["Key"]: row["Value"]
            for row in csv.DictReader(file)
        }

    iso = ast.literal_eval(data["isotherm"])

    ax.errorbar(
        iso["pressure"],
        iso["loading_absolute_average"],
        yerr=iso["loading_absolute_dev"],
        label=label,
        color=color,
        marker=marker,
        linestyle="-",
        linewidth=1.6,
        markersize=5,
        capsize=3,
        elinewidth=1,
        capthick=1,
    )

ax.set(
    xlabel="Pressure (bar)",
    ylabel=r"Absolute loading (mol kg$^{-1}$)",
)

ax.legend(frameon=False)

ax.set_axisbelow(True)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()


# fig.savefig("isotherms_comparison.pdf", bbox_inches="tight")
# fig.savefig("isotherms_comparison.png", dpi=300, bbox_inches="tight")

plt.show()

Here $CO_2$ loading is higher than $CH_4$ loading throughout the pressure
range. The $CH_4$ isotherm is approximately linear, whereas the $CO_2$ isotherm
 is curved. Neither isotherm reaches a saturation plateau
within this pressure range.

## 5. Binary-mixture isotherms using IAST

Conditions : $T = 300\,\mathrm{K}$, $y_{\mathrm{CO_2}}=0.40$ and $y_{\mathrm{CH_4}}=0.60$ (biogas composition).

In [ ]:
import numpy as np
import pandas as pd
import pyiast

isotherms = []
for filename in ["co2_results.csv", "ch4_results.csv"]:
    table = pd.read_csv(filename, index_col="Key")
    data = ast.literal_eval(table.loc["isotherm", "Value"]) #converts the dictionary stored as text into a python dictionary
    
    pure_gas = pd.DataFrame({"pressure": data["pressure"], "loading": data["loading_absolute_average"],})
    
    isotherm = pyiast.InterpolatorIsotherm(pure_gas, pressure_key="pressure", loading_key="loading") #calls the IAST intepolator which basically connects the puregas data points with straight lines
    isotherms.append(isotherm)

pressures = np.linspace(0.1, 3.9, 20)
pressures = np.append(pressures, 4.0) #add the last pressure point since the prevoius one ends with 3.9
y = np.array([0.40, 0.60]) #store the given gas fractions

co2_loadings = []
ch4_loadings = []
for pressure in pressures:
    partial_pressures = pressure * y
    co2_loading, ch4_loading = pyiast.iast(partial_pressures, isotherms) #call the IAST function which calculates the loadings based on the partial pressures and isotherms
    co2_loadings.append(co2_loading)
    ch4_loadings.append(ch4_loading)

In [ ]:
plt.rcdefaults()

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
})

# Same colors and markers as the pure-gas plot.
ch4_color = plt.colormaps["magma"](0.25)
co2_color = plt.colormaps["magma"](0.75)

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

ax.plot(
    pressures,
    ch4_loadings,
    label=r"$\mathrm{CH}_4$",
    color=ch4_color,
    marker="o",
    linestyle="-",
    linewidth=1.6,
    markersize=5,
)

ax.plot(
    pressures,
    co2_loadings,
    label=r"$\mathrm{CO}_2$",
    color=co2_color,
    marker="s",
    linestyle="-",
    linewidth=1.6,
    markersize=5,
)

ax.set(
    xlabel="Total pressure (bar)",
    ylabel=r"Mixture loading (mol kg$^{-1}$)",
    title=(
        r"$T = 300\,\mathrm{K}$, "
        r"$y_{\mathrm{CO}_2} = 0.40$, "
        r"$y_{\mathrm{CH}_4} = 0.60$"
    ),
)

ax.legend(frameon=False)
ax.set_axisbelow(True)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()

# fig.savefig("iast_mixture.pdf", bbox_inches="tight")
# fig.savefig("iast_mixture.png", dpi=300, bbox_inches="tight")

plt.show()

## 6. Selectivity of $CO_2$ over $CH_4$

The selectivity is calculated from the IAST binary loadings at 300 K for the biogas composition ($y_{CO_2}=0.4$, $y_{CH_4}=0.6$):
$$S_{CO_2/CH_4} = \frac{q_{CO_2}}{q_{CH_4}}\frac{y_{CH_4}}{y_{CO_2}}$$

In [ ]:
def selectivity_table(y_co2, pressures=(0.1, 1, 2, 3)):
    """IAST binary loadings (mmol/g) and CO2/CH4 selectivity at the given total pressures (bar)."""
    rows = []
    for P in pressures:
        q_co2, q_ch4 = pyiast.iast([y_co2 * P, (1 - y_co2) * P], isotherms)
        rows.append({"P (bar)": P, "q_CO2 (mmol/g)": q_co2, "q_CH4 (mmol/g)": q_ch4,
                     "S_CO2/CH4": q_co2 / q_ch4 * (1 - y_co2) / y_co2})
    return pd.DataFrame(rows)


selectivity_table(0.4).round(3)

The selectivity is about 4.4–4.5 at all four pressures. $CO_2$ is preferentially adsorbed because it interacts more strongly with the framework (Widom adsorption energy −9.2 vs −3.3 kJ/mol). The selectivity is practically independent of pressure and equal to the limiting selectivity $S^0_{CO_2/CH_4} = 4.42$ from Q3, because up to 3 bar the loadings are far below saturation (Henry regime). A selectivity of about 4.5 is only moderate, so IRMOF-1 is not a very selective adsorbent for $CO_2/CH_4$ separation.

Babarao et al. (2007) found the same trend: the selectivity in IRMOF-1 equals the Henry-constant ratio at low pressure and stays nearly constant before increasing at higher pressures. Their ratio, $K_{H,CO_2}/K_{H,CH_4} = 0.0093/0.0046 \approx 2.0$, is lower than ours (4.42), most likely because they used a different force field (UFF with DFT-derived framework charges).

## 7. Effect of the gas composition on the selectivity

In IAST, each component of the mixture satisfies
$$P\,y_i = P_i^0\,x_i ,$$
where $x_i$ is the mole fraction of $i$ in the adsorbed phase. $P_i^0$ is the pressure of pure $i$ at which it has the same reduced spreading pressure as the mixture:
$$\frac{\pi A}{RT} = \int_0^{P_{CO_2}^0} \frac{q_{CO_2}^0(P)}{P}\,dP = \int_0^{P_{CH_4}^0} \frac{q_{CH_4}^0(P)}{P}\,dP .$$
Here IAST only needs the pure CO$_2$ isotherm up to $P_{CO_2}^0 \approx 1.6$ bar, below the pressures where it curves (Q4). The CH$_4$ isotherm is linear over the whole range. Both can therefore be written as $q_i^0 = K_{H,i}\,P$, and the integrals give $K_{H,CO_2}P_{CO_2}^0 = K_{H,CH_4}P_{CH_4}^0$. Using $q_i = x_i\,q_{tot}$:
$$S_{CO_2/CH_4} = \frac{q_{CO_2}}{q_{CH_4}}\frac{y_{CH_4}}{y_{CO_2}} = \frac{x_{CO_2}}{x_{CH_4}}\frac{y_{CH_4}}{y_{CO_2}} = \frac{P_{CH_4}^0}{P_{CO_2}^0} = \frac{K_{H,CO_2}}{K_{H,CH_4}}$$
The selectivity depends only on the ratio of the Henry coefficients and not on $y_{CO_2}$, so lowering $y_{CO_2}$ from 0.4 to 0.2 leaves it unchanged. The IAST results confirm this:

In [ ]:
comparison = selectivity_table(0.4)[["P (bar)", "S_CO2/CH4"]].rename(columns={"S_CO2/CH4": "S (y_CO2 = 0.4)"})
comparison["S (y_CO2 = 0.2)"] = selectivity_table(0.2)["S_CO2/CH4"]
comparison.round(3)

At every pressure, the selectivity at $y_{CO_2}=0.2$ differs from the one at $y_{CO_2}=0.4$ by less than 1 %, which is within the statistical uncertainty of the GCMC isotherms. Babarao et al. (2007) likewise report that the selectivity is only a weak function of the gas composition.

**Temperature dependence.** Recompute the pure CO$_2$ and CH$_4$ isotherms at several other temperatures (e.g. 250 K and 350 K) with the same simulation parameters. Then repeat the IAST calculation and compare $S_{CO_2/CH_4}$ at the same pressures and composition.